# 모멘텀



## 모멘텀(Momentum) 이란?

모멘텀은 이전 업데이트 방향을 일부 기억해서, 현재 gradient와 함께 파라미터를 이동시키는 방법입니다.
쉽게 말해, 경사하강법에 "관성"을 추가해 진동을 줄이고 더 빠르게 수렴하도록 돕습니다.

## 수식

기본 SGD:

- $\theta_{t+1} = \theta_t - \eta \nabla J(\theta_t)$

Momentum SGD:

- $v_{t+1} = \mu v_t - \eta \nabla J(\theta_t)$
- $\theta_{t+1} = \theta_t + v_{t+1}$

기호 설명:

- $\theta$: 모델 파라미터
- $\eta$: 학습률(learning rate)
- $\nabla J(\theta_t)$: 현재 시점 gradient
- $v$: 속도(누적 업데이트 방향)
- $\mu$: 모멘텀 계수(보통 0.9 근처)

## 장점

- 경사 방향이 일관된 구간에서 수렴 속도를 높임
- 지그재그 진동을 줄여 학습을 안정화함
- 지역적인 작은 요철(noise)에 덜 민감함

## 단점

- 하이퍼파라미터(`lr`, `momentum`) 튜닝이 필요함
- 모멘텀이 너무 크면 overshooting(최소점을 지나침) 가능
- 데이터/모델에 따라 AdamW 같은 적응형 옵티마이저보다 불리할 수 있음

## 역사

- 1964년 Polyak의 "Heavy Ball" 방법에서 아이디어가 출발
- 이후 신경망 최적화에서 SGD 가속 기법으로 널리 사용
- 2013년 Nesterov Accelerated Gradient(NAG)가 딥러닝에 본격 확산

## 사용 방법 요약

1. `optim.SGD(..., momentum=0.9)`처럼 momentum 값을 설정
2. 일반 학습 루프(`zero_grad -> backward -> step`)는 동일
3. 보통 `lr`와 함께 튜닝하며 시작값으로 `momentum=0.9`를 자주 사용

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# 재현성을 위한 시드 고정
torch.manual_seed(42)

# 간단한 선형 데이터: y = 2x + 1
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[3.0], [5.0], [7.0], [9.0]])

# 모델: 1차원 선형회귀
model = nn.Linear(1, 1)

# 핵심: SGD에 momentum 추가
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

print("초기 weight, bias:", model.weight.item(), model.bias.item())

for epoch in range(11):  # 0~10 epoch
    pred = model(x)
    loss = F.mse_loss(pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch in (0, 5, 10):
        print(
            f"Epoch {epoch:>2} | loss={loss.item():.6f} | "
            f"w={model.weight.item():.4f}, b={model.bias.item():.4f}"
        )

# 학습 후 예측 확인
with torch.no_grad():
    test_x = torch.tensor([[5.0]])
    test_pred = model(test_x).item()
    print(f"x=5.0 예측값: {test_pred:.4f}")

초기 weight, bias: 0.7645385265350342 0.8300079107284546
Epoch  0 | loss=12.526728 | w=0.9584, b=0.8952
Epoch  5 | loss=2.468003 | w=2.6556, b=1.4600
Epoch 10 | loss=0.779799 | w=1.9431, b=1.1958
x=5.0 예측값: 10.9115
